In [3]:
# reading the data from parquet file
df_parquet=spark.read.format("parquet").load("/path/to/save/joined_df.parquet")

NameError: name 'spark' is not defined

In [5]:
from pyspark.sql.functions import avg
# Grouping by the "HH_ID" column and counting the number of cars for each HH_ID
average_cars_per_household = df_parquet.groupBy("HH_ID").agg(avg("CAR_ID").alias("avg_cars"))
# Displaying the count of cars by their HH_ID
display(average_cars_per_household)

NameError: name 'df_parquet' is not defined

In [ ]:
from pyspark.sql.functions import col
# Grouping by the "model_year" column and counting the number of cars for each model_year
cars_count_by_model_year = df_parquet.groupBy("Model Year").agg(count("CAR_ID").alias("cars_count"))
# Displaying the count of cars by their Model Year
display(cars_count_by_model_year)

In [ ]:
from pyspark.sql.functions import col

# Grouping by the "MAKE" column and counting the number of cars for each make
cars_count_by_make = df_parquet.groupBy("MAKE").agg(count("CAR_ID").alias("cars_count"))

# Displaying the count of cars by their make
display(cars_count_by_make.orderBy(col("cars_count").desc()))

In [ ]:
from pyspark.sql.functions import col

# Assuming cars with Driver Safety Discount and Vehicle Safety Discount of 1 are considered safe
safe_cars = df_parquet.filter((col("Driver Safety Discount") == 1) & (col("Vehicle Safety Discount") == 1)).select("CAR_ID").distinct()

# Displaying the attributes of safe cars
display(safe_cars)

In [ ]:
from pyspark.sql.functions import count

# Grouping by state and household_id to count the number of customers in each household
household_sizes = df_parquet.groupBy("State", "HH_ID").agg(count("CUST_ID").alias("num_customers"))

# Finding the maximum household size in each state
max_household_sizes = household_sizes.groupBy("State").max("num_customers").alias("max_customers")

# Displaying the states with the largest households
display(max_household_sizes)

In [1]:
import pandas as pd

# Convert 'HH Start Date' to datetime format once before applying the filter
df_parquet["HH Start Date"] = pd.to_datetime(df_parquet["HH Start Date"], format="%Y-%m-%d", errors='coerce')

# Filtering for active households where "Active HH" is 1 and "HH Start Date" is on or before "2021-01-01"
active_households = df_parquet[(df_parquet["Active HH"] == 1) & 
                               (df_parquet["HH Start Date"] <= "2021-01-01")]

# Display the count of active households
print(active_households.shape[0])  # Using .shape[0] to get the number of rows


NameError: name 'df_parquet' is not defined

In [ ]:
from pyspark.sql.functions import datediff, current_date, floor, col, stddev

# Calculating age from DOB considering the leap year for every 4 years
df_with_age = df_parquet.withColumn("age", floor(datediff(current_date(), col("DOB")) / 365.25))

# Grouping by state and calculating the standard deviation of age
age_variance_by_state = df_with_age.groupBy("State").agg(stddev("age").alias("age_variance"))

display(age_variance_by_state)

In [ ]:
from pyspark.sql.functions import when, sum

# Categorizing age into groups
df_with_age_group = df_parquet.withColumn(
    "age_group",
    when(col("age") < 18, "0-17")
    .when((col("age") >= 18) & (col("age") <= 24), "18-24")
    .when((col("age") >= 25) & (col("age") <= 34), "25-34")
    .when((col("age") >= 35) & (col("age") <= 44), "35-44")
    .when((col("age") >= 45) & (col("age") <= 54), "45-54")
    .when((col("age") >= 55) & (col("age") <= 64), "55-64")
    .otherwise("65+")
)

# Summing up claim amounts by age group
claims_by_age_group = df_with_age_group.groupBy("age_group").agg(sum("Claim Payout").alias("total_claims"))

# Finding the age group with the most expensive claim
most_expensive_claim_age_group = claims_by_age_group.orderBy(col("total_claims").desc()).limit(1)

display(most_expensive_claim_age_group)